# DuckDB GeoParquet Benchmarks

This notebook compares direct DuckDB queries across GeoParquet layouts: the raw
Microsoft Planetary Computer layout versus stac-hash-sorted layouts produced by
`cosgp convert`. It builds on the same `cosgp.cli.benchmark` framework that
backs the `cosgp benchmark run`/`compare` CLI commands, so results exported here
can be compared with `cosgp benchmark compare` directly.

For a single dataset with a fixed query suite, prefer the CLI:

```sh
uv run cosgp benchmark run my-dataset "optimized/*.parquet"
uv run cosgp benchmark compare benchmark-results/run-a-* benchmark-results/run-b-*
```

This notebook exists for deeper cross-layout comparisons that the CLI's
single-dataset, single-run model doesn't cover: running a fixed query suite
across several dataset variants at once, inspecting Parquet metadata, and
digging into `EXPLAIN ANALYZE` plans.

## Setup

Requires the `benchmark` extra and the `notebooks` dependency group:

```sh
uv sync --extra benchmark --group notebooks
uv run --group notebooks jupyter lab notebooks/duckdb-geoparquet-benchmarks.ipynb
```

In [ ]:
from __future__ import annotations

import subprocess
from dataclasses import dataclass
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from stac_hash import Hasher

from cosgp.cli.benchmark.queries import DEFAULT_REPEATS, QUERIES
from cosgp.cli.benchmark.runner import BenchmarkResult, BenchmarkRunner, sql_literal

## Data Prep

Sync the raw layout from Source Cooperative, then generate hash-sorted variants
locally with `cosgp convert` instead of ad hoc scripts. Two variants isolate the
effect of sorting from the effect of file count:

- `hashed`: default bucket sizing (lets `cosgp convert` pick a natural file count)
- `hashed_matched_file_count`: `--match-file-count`, so it has the same file count
  as `microsoft`, keeping that variable fixed

This can take a while for the full dataset; it only needs to run once.

In [ ]:
SOURCE_DIR = Path('../data/benchmarks/source/mspc-sentinel-2-l2a')
GENERATED_DIR = Path('../data/benchmarks/generated')

HASH_YEAR = 2025  # matches the sample data's Sentinel-2 L2A acquisition year
HASH_START_DATETIME = datetime(HASH_YEAR, 1, 1, tzinfo=UTC)
HASH_END_DATETIME = datetime(HASH_YEAR + 1, 1, 1, tzinfo=UTC)

BENCHMARK_PARAMS = {
    'collection': 'sentinel-2-l2a',
    'id': 'S2B_MSIL2A_20250101T031029_R075_T52VCJ_20250101T050301',
    'start_datetime': datetime(2025, 6, 1, tzinfo=UTC),
    'end_datetime': datetime(2025, 7, 1, tzinfo=UTC),
    'minx': -109.0,
    'miny': 37.0,
    'maxx': -102.0,
    'maxy': 41.0,
    'aoi_wkt': 'POLYGON((-109 37, -102 37, -102 41, -109 41, -109 37))',
    'max_cloud_cover': 20.0,
}

In [ ]:
SOURCE_DIR.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        'aws', 's3', 'sync',
        's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet/mspc-sentinel-2-l2a/',
        str(SOURCE_DIR),
        '--no-sign-request', '--region', 'us-west-2',
        '--endpoint-url', 'https://s3.us-west-2.amazonaws.com',
        '--only-show-errors',
    ],
    check=True,
)

In [ ]:
def convert(outdir: Path, *extra_args: str) -> None:
    if outdir.exists():
        print(f'{outdir} already exists, skipping convert')
        return
    subprocess.run(
        [
            'cosgp', 'convert', str(SOURCE_DIR), str(HASH_YEAR), str(outdir),
            '--no-progress', *extra_args,
        ],
        check=True,
    )


convert(GENERATED_DIR / 'mspc-sentinel-2-l2a-hashed')
convert(GENERATED_DIR / 'mspc-sentinel-2-l2a-hashed-matched', '--match-file-count')

## Dataset Variants

Keep all variants semantically equivalent (same items) so row counts match
across layouts. `REMOTE_DATASETS` points at the same raw layout hosted on
Source Cooperative; there's no hash-sorted layout hosted there yet, so the
remote run below is a latency-inclusive reference point rather than a full
cross-layout comparison.

In [ ]:
DATASETS = {
    'microsoft': f'{SOURCE_DIR}/*.parquet',
    'hashed': f'{GENERATED_DIR}/mspc-sentinel-2-l2a-hashed/*.parquet',
    'hashed_matched_file_count': f'{GENERATED_DIR}/mspc-sentinel-2-l2a-hashed-matched/*.parquet',
}

SOURCE_COOPERATIVE_PREFIX = 's3://us-west-2.opendata.source.coop/developmentseed/stac-geoparquet'
REMOTE_DATASETS = {
    'remote_microsoft': f'{SOURCE_COOPERATIVE_PREFIX}/mspc-sentinel-2-l2a/*.parquet',
}

DATASETS, REMOTE_DATASETS

## Query Suite

`QUERIES` comes straight from `cosgp.cli.benchmark.queries`, so the notebook
and CLI use the same benchmark definitions. The notebook owns scenario-specific
parameter selection, including `collection` and the hash range used by the
hash-sorted queries.

- `q04_stac_search_count`: a combined collection + time + bbox filter, simulating a
  realistic STAC API `/search`
- `q05_hash_range_search`: the same search, filtered to a hash range instead of
  scanning the full bbox/time predicate — only meaningful for hash-sorted data
- `q06_search_page_hash_order`: a search page ordered by hash instead of datetime

The two hash-only queries are skipped for datasets without a `hash:hash` column.

In [ ]:
HASH_ONLY_QUERIES = {'q05_hash_range_search', 'q06_search_page_hash_order'}
ALL_QUERIES = QUERIES

## Helpers

`BenchmarkRunner` (from the CLI framework) owns the DuckDB connection, query
timing, and `run.json` export. This notebook owns the fixed benchmark scenario
in `BENCHMARK_PARAMS`; for hash-sorted datasets it adds a real hash range for
that query window using `stac_hash.Hasher`, the same hasher `cosgp convert`
uses.

In [ ]:
hasher = Hasher(HASH_START_DATETIME, HASH_END_DATETIME)


def dataset_columns(runner: BenchmarkRunner, path: str) -> set[str]:
    cursor = runner.connection.execute(
        f'DESCRIBE SELECT * FROM read_parquet({sql_literal(path)}) LIMIT 0'
    )
    return {row[0] for row in cursor.fetchall()}


def resolve_hash_range(params: dict[str, object]) -> tuple[int, int]:
    start = params['start_datetime'].astimezone(UTC)
    end = params['end_datetime'].astimezone(UTC)
    corners = [
        (params['minx'], params['miny']),
        (params['maxx'], params['miny']),
        (params['maxx'], params['maxy']),
        (params['minx'], params['maxy']),
    ]
    hashes = [
        hasher.hash_clamped(t, x, y) for t in (start, end) for x, y in corners
    ]
    return min(hashes), max(hashes)


def resolve_all_params(
    runner: BenchmarkRunner, path: str
) -> tuple[dict[str, object], bool]:
    params = dict(BENCHMARK_PARAMS)
    has_hash = 'hash:hash' in dataset_columns(runner, path)
    if has_hash:
        min_hash, max_hash = resolve_hash_range(params)
        params['min_hash'] = min_hash
        params['max_hash'] = max_hash
    return params, has_hash

## Parquet Metadata

Run this before timing queries. It verifies file counts, row groups, and
confirms the hash column is actually present and sorted where expected.

In [ ]:
metadata_sql = """
SELECT
    file_name,
    count(DISTINCT row_group_id) AS row_groups,
    max(row_group_num_rows) AS max_row_group_rows,
    sum(row_group_compressed_bytes) AS compressed_bytes
FROM parquet_metadata({parquet_glob})
GROUP BY file_name
ORDER BY file_name
"""

runner = BenchmarkRunner(repeats=DEFAULT_REPEATS, progress=False)

for name, glob in DATASETS.items():
    print('\n##', name)
    sql = metadata_sql.format(parquet_glob=sql_literal(glob))
    display(runner.connection.execute(sql).df())

## Run Benchmarks

In [ ]:
@dataclass
class DatasetRun:
    dataset_name: str
    dataset_path: str
    params: dict[str, object]
    results: list[BenchmarkResult]

In [ ]:
def run_dataset(
    runner: BenchmarkRunner, dataset_name: str, dataset_path: str
) -> DatasetRun:
    params, has_hash = resolve_all_params(runner, dataset_path)
    results = []
    for query in ALL_QUERIES:
        if query.name in HASH_ONLY_QUERIES and not has_hash:
            continue
        try:
            result = runner.run_query(query, dataset_path, params)
        except Exception as error:
            print(f'{dataset_name} / {query.name}: {error}')
            continue
        results.append(result)
        print(
            f'{dataset_name} / {result.query}: rows={result.rows} '
            f'best={result.best_seconds:0.4f}s median={result.median_seconds:0.4f}s'
        )
    return DatasetRun(dataset_name, dataset_path, params, results)


def run_matrix(runner: BenchmarkRunner, datasets: dict[str, str]) -> list[DatasetRun]:
    return [
        run_dataset(runner, name, path) for name, path in datasets.items()
    ]


def results_table(dataset_runs: list[DatasetRun]) -> pd.DataFrame:
    rows = [
        {
            'query': result.query,
            'dataset': run.dataset_name,
            'rows': result.rows,
            'best_seconds': result.best_seconds,
            'median_seconds': result.median_seconds,
        }
        for run in dataset_runs
        for result in run.results
    ]
    if not rows:
        return pd.DataFrame(
            columns=['query', 'dataset', 'rows', 'best_seconds', 'median_seconds']
        )
    return pd.DataFrame(rows).sort_values(['query', 'median_seconds']).reset_index(drop=True)

In [ ]:
runs = run_matrix(runner, DATASETS)
summary = results_table(runs)
display(summary)

### Remote Object-Store Run

Includes object-store listing, network latency, and HTTP range reads, so
compare it separately from the local-first results above. Set `RUN_REMOTE =
True` to run it; it's off by default since it needs network access and takes
longer.

In [ ]:
RUN_REMOTE = False

if RUN_REMOTE:
    remote_runs = run_matrix(runner, REMOTE_DATASETS)
    remote_summary = results_table(remote_runs)
    display(remote_summary)

## Analyze Results

The speedup table compares median runtimes by query. `hashed` isolates the
sort effect together with whatever file count `cosgp convert` naturally
picked; `hashed_matched_file_count` isolates the sort effect alone by keeping
the file count equal to `microsoft`.

In [ ]:
pivot = summary.pivot(index='query', columns='dataset', values='median_seconds')
speedups = pivot.copy()
if {'microsoft', 'hashed_matched_file_count'}.issubset(speedups.columns):
    speedups['microsoft_vs_hashed_matched_speedup'] = (
        speedups['microsoft'] / speedups['hashed_matched_file_count']
    )
if {'hashed', 'hashed_matched_file_count'}.issubset(speedups.columns):
    speedups['hashed_vs_hashed_matched_speedup'] = (
        speedups['hashed'] / speedups['hashed_matched_file_count']
    )

display(speedups.reset_index())

In [ ]:
metadata_rows = []
for dataset, glob in DATASETS.items():
    sql = metadata_sql.format(parquet_glob=sql_literal(glob))
    df = runner.connection.execute(sql).df()
    metadata_rows.append({
        'dataset': dataset,
        'files': len(df),
        'row_groups': int(df['row_groups'].sum()),
        'compressed_gb': float(df['compressed_bytes'].sum() / 1_000_000_000),
        'median_row_groups_per_file': float(df['row_groups'].median()),
    })

file_summary = pd.DataFrame(metadata_rows).sort_values('dataset').reset_index(drop=True)
display(file_summary)

### Reading the Summary

- `microsoft` vs `hashed_matched_file_count` isolates the sort effect while
  keeping file count fixed.
- `hashed` vs `hashed_matched_file_count` shows how much file count itself
  changes timing for the same hash-sorted data.
- `q01_full_dataset_count` is mostly a raw-scan control; a big difference here
  usually means compression/schema/row-group overhead, not sorting.
- `q04_stac_search_count` is the main STAC-style query to watch; a hashed win
  here is the strongest signal that sorting helps.
- `q05_hash_range_search` and `q06_search_page_hash_order` are only meaningful
  relative to `q04_stac_search_count`/`q06_search_page_datetime_order` on the same
  hash-sorted dataset; they don't run at all on `microsoft`.

Use `median_seconds` for comparisons; `best_seconds` is useful for spotting
warm-cache potential but can be optimistic.

## Inspect a Plan

Use this when a timing difference looks interesting. The key line to look
for is `Total Files Read`; if that number drops, DuckDB is pruning files.
Compares the STAC-style search against its hash-range equivalent on the
same hash-sorted dataset.

In [ ]:
dataset_name = 'hashed_matched_file_count'
run = next(run for run in runs if run.dataset_name == dataset_name)

for query in ALL_QUERIES:
    if query.name not in ('q04_stac_search_count', 'q05_hash_range_search'):
        continue
    sql = query.sql.format(
        **{
            key: sql_literal(value)
            for key, value in {'parquet_glob': DATASETS[dataset_name], **run.params}.items()
        }
    )
    print('\n##', query.name)
    print(sql)
    for row in runner.connection.execute('EXPLAIN ANALYZE ' + sql).fetchall():
        print(row[1] if len(row) > 1 else row[0])

## Export Results

Writes one `run.json` per dataset in the same format `cosgp benchmark run`
produces, so any pair can be compared directly with `cosgp benchmark
compare` — including comparisons against a run produced entirely by the
CLI rather than this notebook.

In [ ]:
out_dir = Path('../benchmark-results')
out_dir.mkdir(parents=True, exist_ok=True)

for run in runs:
    run_file = runner.write(run.dataset_name, run.dataset_path, run.params, run.results, out_dir)
    print(f'wrote {run_file}')

if RUN_REMOTE:
    for run in remote_runs:
        run_file = runner.write(run.dataset_name, run.dataset_path, run.params, run.results, out_dir)
        print(f'wrote {run_file}')